# Cursul 3. Surse diverse și preprocesare
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

**Preprocesarea înseamnă să aducem surse diferite la aceeași formă.** Datele de azi vin în trei feluri: dintr-un API (Eurostat), dintr-un fișier CSV de la o adresă (GISCO) și dintr-un depozit (SCI). Nu judecăm încă dacă sunt bune; asta e curățarea, în C4. Azi doar le aliniem, cu patru operații:

| Operația | În pandas |
|---|---|
| 1. Păstrăm doar coloanele de care avem nevoie | `tabel[["a", "b"]]` |
| 2. Le dăm nume clare, aceleași peste tot | `tabel.rename(columns={...})` |
| 3. Păstrăm doar unitatea de analiză (de exemplu țări, nu grupuri de țări) | un filtru cu o condiție |
| 4. Punem tabelele de aceeași formă unul sub altul | `pd.concat`, cu o coloană care spune de unde vine fiecare rând |

Pe drum observăm și ce rămâne de curățat în C4: valorile lipsă și perechile care apar de două ori. La final, echipa își aduce depozitul proiectului.

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Funcțiile din C2, într-un fișier `.py` | 10 min |
| 2 | Ce am adus din C2 | 5 min |
| 3 | Numele regiunilor: nomenclatorul NUTS | 10 min |
| 4 | Mai mulți indicatori regionali, într-un singur tabel | 15 min |
| 5 | SCI pe regiuni: nume clare | 10 min |
| 6 | Perechi de țări: migrația și zborurile | 20 min |
| 7 | Echipa: depozitul proiectului | 25 min |

Datele merg într-un singur flux: ce am adus de la sursă stă în `data/raw` (din C2, și pe GitHub), iar tabelele curate de azi merg în `data/processed`, câte un fișier pe tabel. În C4 le luăm de acolo și le unim.

## 1. Funcțiile din C2, într-un fișier `.py`
În C2 am scris `cere_eurostat` în notebook. Ca să o folosim în orice notebook, și mai târziu în proiect, o punem într-un fișier de cod, `src/date_api.py`, ca în proiectul echipei: sunt funcțiile din C2, plus `cere_perechi`, bucla din C2 pentru perechi de țări.

Fișierul stă în depozitul cursului. Îl aducem cu `requests`, ca în C2, și îl scriem în `curs/src/`:

In [ ]:
import requests
from pathlib import Path

adresa = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/src/date_api.py"
raspuns = requests.get(adresa)                        # cerem fișierul, ca în C2

Path("src").mkdir(exist_ok=True)                      # folderul curs/src/
Path("src/date_api.py").write_bytes(raspuns.content)  # îl salvăm exact cum a venit

Deschide `src/date_api.py` în VS Code și citește-l: sunt aceleași funcții. `from src.date_api import ...` înseamnă: din fișierul `date_api.py` din folderul `src`, ia funcțiile numite.

In [ ]:
import pandas as pd
from src.date_api import cere_eurostat, cere_perechi   # funcțiile din src/date_api.py

Path("data/processed").mkdir(exist_ok=True)            # tabelele curate, un singur folder pentru tot cursul

Funcția se folosește exact ca în C2: densitatea tuturor regiunilor NUTS2, într-un rând.

In [ ]:
cere_eurostat("demo_r_d3dens", {"geoLevel": "nuts2", "time": 2024}).head()

Dacă apare `No module named 'src'`, verifică două lucruri: celula de mai sus a scris `curs/src/date_api.py`, iar notebook-ul rulează din `curs/` (`Path.cwd()`, ca în C1). Dacă schimbi ceva în `src/date_api.py`, repornește kernelul (**Restart**).

## 2. Ce am adus din C2
Caietul 2 a salvat în `data/raw` câte un fișier pe indicator și nivel, exact cum au venit de la Eurostat. Ca să lucrăm toți pe aceleași date, chiar dacă cineva n-a rulat C2 sau a primit alte valori, le citim din depozitul cursului: sunt exact fișierele salvate de caietul 2, puse pe GitHub, în folderul [`curs/data/raw`](https://github.com/dsw-ubb-2026/-DSW2026/tree/main/curs/data/raw) (deschide linkul și vezi-le):

| Fișierul | Ce conține | Un rând = |
|---|---|---|
| `densitate_nuts2.csv`, `densitate_nuts3.csv` | locuitori pe km², 2024 | o regiune / un județ |
| `populatie_nuts2.csv`, `populatie_nuts3.csv` | numărul de locuitori, 2024 | o regiune / un județ |
| `pib_locuitor_nuts2.csv`, `pib_locuitor_nuts3.csv` | PIB pe locuitor, în PPS, 2024 | o regiune / un județ |
| `migratie.csv` | rezidenți după cetățenie, 2024 | o țară de rezidență și o cetățenie |
| `bm.csv` | indicatorii Băncii Mondiale, 2022-2024 | o țară, un an și un indicator |

`DATE_C2` e adresa **folderului**. Un fișier se citește lipind la ea numele lui: `DATE_C2 + "densitate_nuts3.csv"`. Doar `pd.read_csv(DATE_C2)` nu merge, pentru că un folder nu e un tabel.

In [ ]:
DATE_C2 = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/raw/"   # fișierele din C2

**Anticipează:** ce coloane are unul dintre ele?

In [ ]:
densitate = pd.read_csv(DATE_C2 + "densitate_nuts3.csv")    # adresa + numele fișierului
densitate.head()

Două coloane: codul regiunii (`geo`) și valoarea. Un rând = o regiune. Clujul îl găsim după cod, cu filtrul din C1:

In [ ]:
densitate[densitate["geo"] == "RO113"]      # rândurile în care geo este RO113

**Exercițiu.** Câți locuitori are regiunea Nord-Vest (`RO11`)? Fișierul este `populatie_nuts2.csv`.

In [ ]:
# 1. Citește DATE_C2 + "populatie_nuts2.csv" în variabila populatie
# 2. Rândul în care geo este "RO11", ca la densitate

Codul `RO11` nu spune singur că e vorba de Nord-Vest. Numele vin din nomenclatorul NUTS.

## 3. Numele regiunilor: nomenclatorul NUTS
*Operațiile 1 și 2: coloanele de care avem nevoie, cu nume clare.*

Eurostat publică lista tuturor regiunilor (codul, țara, numele) ca fișier CSV, pe serverul GISCO ([pagina fișierelor](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)). `read_csv` îl citește direct de la adresa lui, ca pe un fișier de pe calculator.

In [ ]:
nuts = pd.read_csv("https://gisco-services.ec.europa.eu/distribution/v2/nuts/csv/NUTS_AT_2024.csv")
nuts.head()

Păstrăm trei coloane și le dăm nume în română, aceleași în tot proiectul. Lungimea codului spune nivelul: `RO` e țara, `RO11` regiunea (NUTS2), `RO113` județul (NUTS3).

In [ ]:
nuts = nuts[["NUTS_ID", "CNTR_CODE", "NAME_LATN"]]   # codul, țara, numele cu litere latine
nuts.columns = ["regiune", "tara", "denumire"]       # nume noi, în aceeași ordine
nuts["nivel"] = nuts["regiune"].str.len() - 2        # RO -> 0, RO11 -> 2, RO113 -> 3
nuts[nuts["tara"] == "RO"].head(8)

**Exercițiu.** Care sunt regiunile NUTS2 ale României? Două condiții, fiecare între paranteze, legate cu `&`.

In [ ]:
# Rândurile din nuts în care tara este "RO" și nivel este 2

In [ ]:
nuts.to_csv("data/processed/regiuni.csv", index=False)

## 4. Indicatorii regionali, într-un singur tabel
*Operațiile 2 și 4: nume clare, apoi tabelele unul sub altul.*

Din C2 avem populația, densitatea și PIB-ul pe locuitor. Literatura despre SCI arată că legăturile depind și de cât de asemănătoare sunt două regiuni (vârstă, educație, piața muncii) și de cât folosesc oamenii rețelele sociale. Ca toate să aibă aceeași formă, le cerem pe toate șapte din Eurostat, pe regiuni NUTS2, în 2024. Seturile și filtrele sunt în [`CATALOG.md`](https://github.com/dsw-ubb-2026/-DSW2026/blob/main/curs/data/CATALOG.md), în depozitul cursului.

Fiecare rând din listă: numele nostru, setul Eurostat, filtrele.

In [ ]:
INDICATORI = [
    ["populatie",         "demo_r_pjangrp3", {"geoLevel": "nuts2", "time": 2024, "sex": "T", "age": "TOTAL", "unit": "NR"}],
    ["densitate",         "demo_r_d3dens",   {"geoLevel": "nuts2", "time": 2024, "unit": "PER_KM2"}],
    ["pib_locuitor",      "nama_10r_3gdp",   {"geoLevel": "nuts2", "time": 2024, "unit": "PPS_EU27_2020_HAB"}],
    ["varsta_medie",      "demo_r_pjanind2", {"geoLevel": "nuts2", "time": 2024, "indic_de": "MEDAGEPOP", "unit": "YR"}],
    ["studii_superioare", "edat_lfse_04",    {"geoLevel": "nuts2", "time": 2024, "isced11": "ED5-8", "age": "Y25-64", "sex": "T", "unit": "PC"}],
    ["somaj",             "lfst_r_lfu3rt",   {"geoLevel": "nuts2", "time": 2024, "isced11": "TOTAL", "age": "Y15-74", "sex": "T", "unit": "PC"}],
    ["retele_sociale",    "isoc_r_iuse_i",   {"geoLevel": "nuts2", "time": 2024, "indic_is": "I_IUSNET", "unit": "PC_IND"}],
]

Aceiași doi pași pentru fiecare indicator: cerem tabelul și notăm ce măsoară. La final îi punem unul sub altul, cu `pd.concat`, ca la Banca Mondială în C2.

In [ ]:
tabele = []
for nume, dataset, filtre in INDICATORI:
    tabel = cere_eurostat(dataset, filtre)   # un indicator, toate regiunile NUTS2
    tabel["indicator"] = nume                # ce măsoară valoarea
    tabele.append(tabel)

indicatori = pd.concat(tabele)                                 # cele 7 tabele, unul sub altul
indicatori = indicatori.rename(columns={"geo": "regiune"})     # același nume ca în nomenclator
indicatori.shape

Tabelul e în format **lung**: o regiune apare de mai multe ori, câte un rând pe indicator.

**Exercițiu.** Afișează toate rândurile regiunii Nord-Vest (`RO11`).

In [ ]:
# Rândurile din indicatori în care regiune este "RO11"

**Pentru C4: valorile lipsă.** Câte rânduri are fiecare indicator și câte au valoare? `size` numără rândurile, `count` doar valorile prezente; diferența sunt valorile lipsă.

In [ ]:
indicatori.groupby("indicator")["valoare"].agg(["size", "count"])

**Interpretează:** unde lipsesc cele mai multe valori? O valoare lipsă nu înseamnă zero: sursa nu are date pentru acea regiune în 2024. Ce facem cu ele decidem în C4.

In [ ]:
indicatori.to_csv("data/processed/indicatori.csv", index=False)

## 5. SCI pe regiuni: nume clare
*Operația 2: nume clare.*

SCI există și pe regiuni NUTS2, din aceeași sursă ca `country.csv` din C1 ([HDX](https://data.humdata.org/dataset/social-connectedness-index)). Fișierul stă în depozitul cursului; îl citim direct de la adresa lui, ca nomenclatorul.

In [ ]:
sci = pd.read_csv("https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/nuts2_2024.csv")
sci.head()

Coloanele au nume în engleză. Le dăm nume care spun ce înseamnă în proiect: cele două capete ale perechii.

In [ ]:
nume_noi = {"user_country": "tara_1", "friend_country": "tara_2",
            "user_region": "regiune_1", "friend_region": "regiune_2",
            "scaled_sci": "sci"}
sci = sci.rename(columns=nume_noi)      # se schimbă doar numele din dicționar
print(sci["regiune_1"].nunique(), "regiuni")
print(len(sci), "rânduri")

298 × 298 = 88.804: fiecare regiune cu fiecare, inclusiv cu ea însăși.

**Exercițiu.** Cât este SCI între Nord-Vest (`RO11`) și Lombardia (`ITC4`)? Apoi schimbă ordinea celor două coduri.

In [ ]:
# Rândul din sci în care regiune_1 este "RO11" și regiune_2 este "ITC4"

**Pentru C4: perechile duble.** Aceeași valoare, în ambele ordini: fiecare pereche apare de două ori. În C4 păstrăm o singură apariție.

In [ ]:
sci.to_csv("data/processed/sci_regiuni.csv", index=False)

SCI pe județe (NUTS3) are 1,8 milioane de rânduri, peste 40 MB. Nu îl folosim la curs; echipele care au nevoie de el îl descarcă de pe [HDX](https://data.humdata.org/dataset/social-connectedness-index) și fac aceiași pași.

## 6. Perechi de țări: migrația și zborurile
### 6.1. Migrația: țări, nu grupuri de țări
*Operațiile 1, 2 și 3: coloanele, numele și unitatea de analiză.*

Deschidem tabelul din C2. O capcană: codul Namibiei este `NA`, iar pandas citește `NA` ca valoare lipsă. Îi spunem să considere lipsă doar celulele goale.

In [ ]:
migratie = pd.read_csv(DATE_C2 + "migratie.csv", keep_default_na=False, na_values=[""])
migratie["citizen"].unique()

**Anticipează:** care coduri nu sunt țări? Eurostat pune în aceeași coloană și grupuri: `EU27_2020_FOR` (străinii din UE), `FOR` (toți străinii), `NAT` (cetățenii proprii), `TOTAL`. Dacă le păstrăm, aceiași oameni sunt numărați de mai multe ori. Codurile de țară au exact două litere; păstrăm rândurile în care **ambele** coloane sunt țări.

In [ ]:
cetatenie_tara = migratie["citizen"].str.len() == 2     # True unde cetățenia e o țară
rezidenta_tara = migratie["geo"].str.len() == 2         # True unde locul e o țară
migratie = migratie[cetatenie_tara & rezidenta_tara]    # amândouă

Scoatem și cetățenii propriei țări (românii din România): nu sunt migranți. Apoi păstrăm trei coloane, cu nume clare.

In [ ]:
migratie = migratie[migratie["citizen"] != migratie["geo"]]      # cetățenia diferă de țară
migratie = migratie[["geo", "citizen", "valoare"]]               # coloanele, în ordinea dorită
migratie.columns = ["tara_rezidenta", "cetatenie", "persoane"]   # nume clare
migratie.to_csv("data/processed/migratie.csv", index=False)

romani = migratie[migratie["cetatenie"] == "RO"]                 # românii, după țara în care trăiesc
romani.sort_values("persoane", ascending=False).head(5)

### 6.2. Zborurile: o legătură vie între țări
*Aceleași operații, pe o sursă nouă.*

Migrația spune câți oameni s-au mutat. Pasagerii aerieni dintre două țări arată cât se circulă între ele: vizite, întoarceri, du-te-vino. Setul este `avia_paocc` (vezi [`CATALOG.md`](https://github.com/dsw-ubb-2026/-DSW2026/blob/main/curs/data/CATALOG.md)).

**Ce înseamnă filtrele.** Ca în C2, fiecare set Eurostat are mai multe dimensiuni, iar fiecare valoare are un cod. Pentru un singur număr pe pereche de țări, fixăm toate dimensiunile, mai puțin cea care variază:

| Dimensiunea | Codul ales | Ce înseamnă |
|---|---|---|
| `unit` | `PAS` | unitatea: pasageri (nu zboruri, nu locuri) |
| `tra_meas` | `PAS_CRD` | ce se măsoară: pasagerii transportați efectiv, cei care au zburat |
| `schedule` | `TOTAL` | toate zborurile: regulate și charter |
| `freq` | `A` | frecvența: anual (nu lunar sau trimestrial) |
| `time` | `2024` | anul |
| `geo` | o țară | țara care raportează (de unde se zboară) |
| `partner` | toate | țara cu care se zboară: asta variază, un rând pe țară parteneră |

Codurile nu se ghicesc. Le găsești în [Eurostat Data Browser](https://ec.europa.eu/eurostat/databrowser/view/avia_paocc/default/table): la fiecare dimensiune, lista arată numele și codul fiecărei valori. Pentru seturile cursului, codurile sunt deja scrise în [`CATALOG.md`](https://github.com/dsw-ubb-2026/-DSW2026/blob/main/curs/data/CATALOG.md).

**Cererea.** Ca la migrație, într-o cerere poate varia un singur lucru, deci cerem țară cu țară. `cere_perechi` face bucla din C2:
1. ia pe rând fiecare țară din listă și o pune la `geo`;
2. cere de la Eurostat toate țările partenere (`pe="partner"`), cu filtrele de mai sus;
3. notează țara în coloana `geo` și lipește tabelele unul sub altul.

Ca listă de țări folosim țările din tabelul migrației: sunt țările pentru care Eurostat are date.

In [ ]:
tari = migratie["tara_rezidenta"].unique()          # țările Eurostat, din migrație

filtre = {"unit": "PAS",            # în pasageri
          "tra_meas": "PAS_CRD",    # pasagerii transportați
          "schedule": "TOTAL",      # zboruri regulate și charter
          "freq": "A",              # anual
          "time": 2024}             # anul

zboruri = cere_perechi("avia_paocc", filtre, tari, pe="partner")   # o cerere pe țară, durează un minut
zboruri.head()

Un rând: câți pasageri au zburat în 2024 între țara `geo` și țara `partner`. Țările fără date apar în listă ca „fără date: ...”; bucla le sare și merge mai departe.

Aceeași curățenie ca la migrație: doar țări, fără perechea unei țări cu ea însăși, cu nume clare.

In [ ]:
zboruri = zboruri[zboruri["partner"].str.len() == 2]             # doar țări, nu grupuri (EU27_2020)
zboruri = zboruri[zboruri["partner"] != zboruri["geo"]]          # fără zborurile interne
zboruri = zboruri[["geo", "partner", "valoare"]]
zboruri.columns = ["tara_1", "tara_2", "pasageri"]
zboruri.to_csv("data/processed/zboruri.csv", index=False)

**Exercițiu.** Cu ce țări are România cei mai mulți pasageri? Primele 10. Unde e Italia, comparativ cu migrația și cu SCI din C1?

In [ ]:
# Rândurile din zboruri în care tara_1 este "RO", ordonate după pasageri, primele 10

**Pentru C4: perechile duble.** Fiecare țară raportează propriile zboruri, deci RO-IT și IT-RO apar separat, cu valori apropiate. În C4 alegem una dintre ele, ca la SCI.

Alte date pe perechi de țări (turiști, comerț, imigranți pe an) sunt în [`CATALOG.md`](https://github.com/dsw-ubb-2026/-DSW2026/blob/main/curs/data/CATALOG.md), cu filtrele lor. Le aduceți cu aceleași funcții.

## 7. Echipa: depozitul proiectului
De azi lucrați și în echipă. Fiecare echipă are pe GitHub depozitul ei, `dsw-ubb-2026/echipa-N`, cu aceeași structură pentru toate echipele. Invitația a venit pe emailul contului GitHub.

**Două locuri, două roluri.** Ce facem la seminar stă în `curs/`, ce faceți pentru proiect stă în `proiect/`. `curs/` e doar al vostru, pe calculator, și nu se urcă nicăieri. `proiect/` e depozitul echipei: doar acolo se fac `pull`, `commit` și `push`. Cele două se ating într-un singur punct: `date_api.py`, copiat o dată din `curs/src/` în `proiect/src/`.

### 7.1. Depozitul pe calculator (fiecare membru)
1. Acceptați invitația: `https://github.com/dsw-ubb-2026/echipa-N/invitations` (N = numărul echipei).
2. Deschideți folderul `dsw` în VS Code (**File → Open Folder**), apoi terminalul (**Terminal → New Terminal**). Terminalul pornește în `dsw/`.
3. Clonați depozitul echipei:
```bash
git clone https://github.com/dsw-ubb-2026/echipa-N.git proiect
```
Cuvântul `proiect` de la final dă numele folderului. Fără el, folderul s-ar numi `echipa-N`.

Prima dată pe un calculator, Git vă cere numele și emailul:
```bash
git config --global user.name "Prenume Nume"
git config --global user.email "emailul-de-pe-github"
```
Rezultatul:
```text
dsw/                      din ZIP, nu e depozit Git
├── .venv/                mediul Python, comun pentru curs și proiect
├── .env                  adresa bazei de date (din C5), nu se urcă
├── curs/                 caietele de seminar, fără Git
└── proiect/              depozitul echipei, cu Git
    ├── data/raw/         datele originale, neschimbate
    ├── data/processed/   tabelele produse de caiete
    ├── notebooks/        câte un caiet pe pas
    ├── src/              cod comun, în fișiere .py
    ├── figures/          figurile pentru raport
    ├── docs/             raportul
    ├── requirements.txt  bibliotecile proiectului
    └── README.md         descrierea proiectului
```

**De evitat:**
- clonarea în `curs/`, pe Desktop sau în alt folder: depozitul stă doar în `dsw/proiect/`;
- `git init` în `dsw/` sau în `curs/`: singurul depozit Git e `proiect/`;
- lucrul pentru proiect în caietele din `curs/`: caietele proiectului le scrieți în `proiect/notebooks/`.

Dacă aveți deja un folder `proiect` cu fișiere în el, `git clone` refuză. Redenumiți-l întâi.

### 7.2. Datele și codul proiectului
Proiectul are același flux ca la curs: în `proiect/data/raw/` ce aduceți de la sursă, exact cum a venit; în `proiect/data/processed/` tabelele curate produse de caiete. Fișierele de peste 50 MB nu se urcă pe GitHub: caietul care le folosește spune de unde se descarcă.

Copiați `curs/src/date_api.py` în `proiect/src/`: e codul comun al echipei. Caietele proiectului stau în `proiect/notebooks/`, un nivel mai jos, deci prima lor celulă îi spune lui Python să caute și un nivel mai sus:
```python
import sys
sys.path.append("..")                       # folderul proiect/, unde e src/
from src.date_api import cere_eurostat
```
VS Code pornește fiecare caiet din folderul lui, deci `..` înseamnă `proiect/` la toți membrii, oriunde ar fi `dsw/` pe calculator.

Căile spre date sunt mereu relative la caiet: `"../data/raw/migratie.csv"`, niciodată `C:\Users\Ana\...`. O cale absolută merge doar pe calculatorul celui care a scris-o.

Mediul Python e același `dsw/.venv` de la curs. Dacă cineva are nevoie de o bibliotecă nouă, o adaugă în `proiect/requirements.txt` și anunță echipa, ca fiecare să ruleze din nou `pip install -r proiect/requirements.txt`.

Ce seturi de date puteți folosi, cu filtrele lor: [`CATALOG.md`](https://github.com/dsw-ubb-2026/-DSW2026/blob/main/curs/data/CATALOG.md).

**Un proiect-model**, cu exact această structură, apare după curs în depozitul cursului, în `proiect-exemplu/`: `notebooks/01_collection.ipynb` aduce toate datele în `data/raw/`, iar `02_preprocessing.ipynb` face din ele tabelele din `data/processed/`. Așa arată pașii 1 și 2 din rolurile echipei.

### 7.3. Reguli de lucru în echipă
Un conflict apare când doi oameni schimbă **același fișier** în paralel. Câteva reguli îl evită:

| Ce | Regula |
|---|---|
| caietele | Fiecare caiet are un responsabil, cel cu rolul pasului, dar toți pot lucra la el. Doar unul câte unul: cine vrea să-l schimbe anunță pe Teams, face `pull`, lucrează, face `push` și spune că a terminat. Abia apoi intră următorul. |
| fișierele de date | Fiecare fișier din `data/raw/` și `data/processed/` e produs de un singur caiet. Ceilalți doar îl citesc. Dacă le trebuie altceva, îi cer responsabilului. |
| numele și coloanele | Se trec în `PROCESSED_DATA.md`: pasul următor se bazează pe ele. Cine redenumește o coloană scrie asta acolo și anunță echipa. |
| caietul unui coleg, doar rulat | Dacă doar îl rulezi ca să vezi ce face, nu îl urca: rularea îi schimbă ieșirile. Înainte de commit, în panoul **Source Control**, alegi **Discard Changes** pe fișierul lui. Pentru încercări proprii, îți faci caietul tău. |
| `README.md` | Îl editează o singură persoană o dată. Vă înțelegeți pe Teams cine. |
| `src/date_api.py` | Se modifică rar și anunțat. După `git pull`, ceilalți dau **Restart** la kernel. |

Ca să lucrați **în același timp** la un pas, aveți trei feluri, fără conflicte:
- **împreună, la un calculator:** unul scrie, ceilalți gândesc cu el; e cel mai simplu pentru o problemă grea;
- **fiecare cu caietul lui de încercări**, de exemplu `03_explorare_ana.ipynb`; ce merge, responsabilul mută în caietul pasului;
- **pe bucăți de cod:** o funcție nouă, scrisă de un coleg în `src/`, pe care caietul pasului doar o folosește.

Și, la fiecare sesiune de lucru: lucrați pe `dev` (7.5), cu `git pull` înainte să începeți și `git push` imediat după, în commit-uri mici.

Pe scurt: la un fișier lucrează o singură persoană o dată, iar fiecare fișier de date are un singur caiet care îl produce.

### 7.4. Primul commit al echipei
Un singur membru completează în `README.md`, provizoriu: titlul, întrebarea de cercetare, unitatea de analiză și tabelul surselor de date. Altul adaugă `src/date_api.py` (`git add src/date_api.py`). Fiecare urcă ce a făcut:
```bash
cd proiect
git pull
git add README.md
git commit -m "Tema si intrebarea provizorie"
git push
```
`commit` salvează pasul pe calculatorul vostru, `push` îl trimite pe GitHub, `pull` îl aduce de la colegi. Comenzile Git se dau doar în `proiect/`: de aceea primul pas e `cd proiect`.

### 7.5. Ramura `dev`
`main` rămâne versiunea stabilă a proiectului. Lucrul de zi cu zi se face pe o a doua ramură, `dev`.

Un singur membru o creează, o singură dată, după primul commit:
```bash
git switch -c dev              # creează dev din main și trece pe ea
git push -u origin dev         # o pune pe GitHub
```
Ceilalți o aduc pe calculatorul lor:
```bash
git fetch                      # află ce ramuri sunt pe GitHub
git switch dev                 # trec pe dev
```
De acum, fiecare sesiune de lucru începe cu `git switch dev` și `git pull`. Când un pas e încheiat și verificat (datele curate, explorarea, modelul), echipa îl aduce în `main`:
```bash
git switch main
git pull
git merge dev                  # aduce în main ce s-a lucrat pe dev
git push
git switch dev                 # înapoi la lucru
```
Toate comenzile, explicate: fișa **Lucrul cu Git** de pe site.

### 7.6. Ce stabiliți azi în echipă
- tema proiectului și o întrebare provizorie: coridorul sau țara pe care o studiați;
- un rol provizoriu pentru fiecare: colectarea, preprocesarea, curățarea și integrarea, explorarea, modelul. Rolurile se pot schimba, iar toți pot colabora la orice pas;
- ce date din catalog vă ajută să răspundeți la întrebare;
- ce reprezintă un rând în tabelul vostru final.

Tema 1, pe Teams, cere fiecăruia să descrie cu propriile cuvinte tema proiectului, datele și rolul ales.

## Ce ducem mai departe
În `data/processed` avem cinci tabele curate: `regiuni.csv` (numele regiunilor), `indicatori.csv` (șapte indicatori pe regiuni), `sci_regiuni.csv`, `migratie.csv` și `zboruri.csv`. Nume clare de coloane, gata de unit în C4. Funcțiile de colectare stau în `src/date_api.py`, iar echipa are depozitul pe calculator și primul commit pe GitHub.

**Poți explica fără cod?** Care sunt cele patru operații ale preprocesării și prin ce diferă de curățare? De ce punem funcțiile într-un fișier `.py`, în `src/`? Ce reprezintă un rând în fiecare tabel? De ce adăugăm coloana `indicator`? De ce `NA` nu e întotdeauna o valoare lipsă? De ce scoatem grupurile de țări? Ce diferență este între `commit`, `push` și `pull`? De ce stă proiectul în `proiect/` și nu în `curs/`? Ce face echipa ca doi membri să nu schimbe același fișier?

### Surse
- [GISCO: fișierele NUTS 2024](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)
- [Social Connectedness Index, pe HDX](https://data.humdata.org/dataset/social-connectedness-index)
- [Eurostat Data Browser](https://ec.europa.eu/eurostat/databrowser/): `migr_pop1ctz`, `avia_paocc`, `demo_r_pjanind2`, `edat_lfse_04`, `lfst_r_lfu3rt`, `isoc_r_iuse_i`
- Bailey, Johnston, Kuchler, Russel, State și Stroebel (2020), *The Determinants of Social Connectedness in Europe*, SocInfo.
- [Git: înregistrarea schimbărilor](https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository)